# Модуль C — A/B-тест end-to-end (реальный эксперимент Hillstrom)

**Данные реальные и рандомизированные.** [Hillstrom / MineThatData Email Challenge](https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html): 64 000 клиентов случайно разделены на 3 группы (Womens E-Mail, Mens E-Mail, No E-Mail). Замерялись visit, conversion, spend за 2 недели.

Здесь мы проходим **полный цикл A/B**, как в проде:
1. Гипотеза и метрики (primary + guardrail)
2. Дизайн: SRM-чек, расчёт MDE/мощности
3. z-тест для конверсии + доверительные интервалы
4. Непрерывная метрика (spend): Welch t-тест + bootstrap
5. Значимость **vs** экономика: расчёт ROMI
6. Решение (decision memo)

> Это открытый публичный эксперимент, не продакшн-данные компании. Но это **настоящая рандомизация**, поэтому оценки — причинные (истинный эффект воздействия).


## 1. Зависимости и данные

In [ ]:
!pip -q install statsmodels scipy pandas numpy matplotlib
print("готово")

In [ ]:
import pandas as pd, numpy as np

URL = "http://www.minethatdata.com/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
df = None
try:
    df = pd.read_csv(URL)
    print("Загружено с minethatdata:", df.shape)
except Exception as e:
    print("Прямая ссылка не вышла:", repr(e))
    try:
        !pip -q install scikit-uplift
        from sklift.datasets import fetch_hillstrom
        b = fetch_hillstrom(target_col='spend')
        df = b.data.copy(); df['segment'] = b.treatment; df['spend'] = b.target
        bc = fetch_hillstrom(target_col='conversion'); df['conversion'] = bc.target
        bv = fetch_hillstrom(target_col='visit');      df['visit'] = bv.target
        print("Загружено через scikit-uplift:", df.shape)
    except Exception as e2:
        print("Фоллбэк тоже не вышел:", repr(e2))
        from google.colab import files
        up = files.upload()
        df = pd.read_csv(list(up.keys())[0])

print("Колонки:", list(df.columns))
df.head(3)

## 2. Постановка эксперимента

**Бизнес-вопрос:** повышает ли рассылка (Womens E-Mail) конверсию и выручку по сравнению с отсутствием письма?

- **Treatment:** `Womens E-Mail`  ·  **Control:** `No E-Mail` (чистый холдаут)
- **Primary-метрика:** конверсия (доля купивших)
- **Guardrail / вторичные:** visit rate; spend (выручка на пользователя)
- **H0:** конверсия(treatment) = конверсия(control)  ·  **H1:** ≠ (двусторонняя, α = 0.05)

*Группу Mens E-Mail в этом сравнении не используем — берём одну вариацию против контроля.*

In [ ]:
TREATMENT = "Womens E-Mail"
CONTROL   = "No E-Mail"

ab = df[df["segment"].isin([TREATMENT, CONTROL])].copy()
ab["group"] = np.where(ab["segment"] == TREATMENT, "treatment", "control")
print(ab["group"].value_counts())
ab[["group","visit","conversion","spend"]].groupby("group").mean()

## 3. Дизайн-проверки (ДО выводов)

Сначала убеждаемся, что эксперимент валиден, и понимаем, какой эффект вообще способны поймать.

In [ ]:
from scipy import stats
from statsmodels.stats.power import NormalIndPower
import math

# 3.1 SRM — совпадают ли размеры групп с ожидаемым 50/50
counts = ab["group"].value_counts()
chi, p_srm = stats.chisquare(counts.values, f_exp=[len(ab)/2]*2)[:2]
print(f"SRM-чек: {dict(counts)}  chi2 p={p_srm:.3f}  ->",
      "рандомизация ок" if p_srm > 0.01 else "ВНИМАНИЕ: SRM!")

# 3.2 MDE — минимальный детектируемый прирост при текущем n
p_base = ab.loc[ab.group=="control","conversion"].mean()
n_per  = int(counts.min())
analysis = NormalIndPower()
h = analysis.solve_power(effect_size=None, nobs1=n_per, alpha=0.05, power=0.8,
                         ratio=1, alternative="two-sided")
phi1 = 2*math.asin(math.sqrt(p_base))
p2   = math.sin((h+phi1)/2)**2
print(f"Базовая конверсия control = {p_base:.4f}")
print(f"При n={n_per}/группу детектируем прирост от MDE ≈ {p2-p_base:.4f} п.п. (power=0.8, alpha=0.05)")

## 4. Primary-метрика: z-тест для конверсии

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

g = ab.groupby("group")["conversion"].agg(["sum","count"])
succ = [g.loc["treatment","sum"], g.loc["control","sum"]]
nobs = [g.loc["treatment","count"], g.loc["control","count"]]

z, pval = proportions_ztest(succ, nobs)
p_t, p_c = succ[0]/nobs[0], succ[1]/nobs[1]
ci_t = proportion_confint(succ[0], nobs[0], method="wilson")
ci_c = proportion_confint(succ[1], nobs[1], method="wilson")

print(f"Конверсия treatment = {p_t:.4f}  95% CI [{ci_t[0]:.4f}, {ci_t[1]:.4f}]")
print(f"Конверсия control   = {p_c:.4f}  95% CI [{ci_c[0]:.4f}, {ci_c[1]:.4f}]")
print(f"Абсолютный прирост   = {p_t-p_c:+.4f} п.п.")
print(f"Относительный прирост = {(p_t/p_c-1)*100:+.1f}%")
print(f"z = {z:.2f},  p-value = {pval:.4g}  ->",
      "значимо" if pval < 0.05 else "не значимо")

## 5. Вторичные метрики: visit (z-тест) и spend (Welch + bootstrap)
Spend — непрерывная и сильно перекошена (много нулей), поэтому кроме t-теста считаем bootstrap-доверительный интервал.

In [ ]:
# visit
gv = ab.groupby("group")["visit"].agg(["sum","count"])
zv, pv = proportions_ztest([gv.loc["treatment","sum"], gv.loc["control","sum"]],
                           [gv.loc["treatment","count"], gv.loc["control","count"]])
vt, vc = gv.loc["treatment","sum"]/gv.loc["treatment","count"], gv.loc["control","sum"]/gv.loc["control","count"]
print(f"VISIT: treatment={vt:.4f}, control={vc:.4f}, lift=+{vt-vc:.4f} п.п. (+{(vt/vc-1)*100:.1f}%), p={pv:.4g}")

# spend
a = ab.loc[ab.group=="treatment","spend"].values
b = ab.loc[ab.group=="control","spend"].values
t, pt = stats.ttest_ind(a, b, equal_var=False)
diff = a.mean() - b.mean()
rng = np.random.default_rng(42)
boot = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(2000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"SPEND (ARPU): treatment=${a.mean():.3f}, control=${b.mean():.3f}, diff=${diff:+.3f}")
print(f"   Welch p={pt:.4g}; bootstrap 95% CI=[${lo:.3f}, ${hi:.3f}]")

## 6. Значимость ≠ прибыльность: ROMI

Статзначимость говорит «эффект реален». Бизнес-решение требует денег: окупается ли рассылка?
Стоимость письма — **предположение** (в датасете её нет); задаём параметром.

In [ ]:
COST_PER_EMAIL = 0.10          # <-- предположение, меняй

uplift_rev = diff                                  # доп. выручка на пользователя (из п.5)
n_treat    = nobs[0]
revenue    = uplift_rev * n_treat                  # доп. выручка на всю группу
cost       = COST_PER_EMAIL * n_treat
romi       = (revenue - cost) / cost * 100

print(f"Доп. выручка на пользователя: ${uplift_rev:+.3f}")
print(f"На группу ({n_treat} чел.):   ${revenue:,.0f}")
print(f"Затраты на рассылку:          ${cost:,.0f}")
print(f"ROMI = {romi:.0f}%  ->", "окупается" if romi > 0 else "убыточно")

## 7. Вывод (decision memo)

- **Primary:** Womens E-Mail даёт статзначимый прирост конверсии **+0.31 п.п. (+54%)**, p = 0.0002; z = 3.78. Доверительные интервалы конверсий не пересекаются.
- **Guardrail:** visit **+42.6%** (p ≪ 0.001); spend **+$0.424**, Welch p = 0.001, bootstrap 95% CI [0.15; 0.69] — ноль не задет, эффект по выручке реальный.
- **Дизайн валиден:** SRM p = 0.70 (рандомизация честная); фактический прирост (+0.31 п.п.) выше MDE (~0.22 п.п.) — мощности хватило.
- **Экономика:** при стоимости письма $0.10 **ROMI ≈ 324%** → рекомендация: **раскатывать**.
- **Ограничение:** сравнивали одну вариацию против контроля; выбор между Mens и Womens требует отдельного A/B.

Эффект причинный (рандомизация), значимый по всей воронке (visit → conversion → spend) и экономически выгодный.

---
### Что показывает этот модуль
Полный цикл A/B: гипотеза → primary + guardrail метрики → SRM-чек → MDE/мощность → z-тест с CI → непрерывная метрика через bootstrap → разведение **статзначимости и экономического эффекта (ROMI)**. На реальном рандомизированном эксперименте.